In [1]:
from google.colab import drive
import pandas as pd
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/map-charting-student-math-misunderstandings')

train_path = DATA_DIR / 'train.csv'
if train_path.exists():
    train_df = pd.read_csv(train_path)
    print(f"成功讀取資料集！共 {len(train_df)} 筆資料。")
    display(train_df.head(2))
else:
    print(f"找不到檔案，請確認路徑 {train_path} 是否正確。")

成功讀取資料集！共 36696 筆資料。


,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception
0,0,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN
1,1,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct,NaN


In [2]:
import pandas as pd
import numpy as np
import pickle
import os
import random
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

print("正在合併文字特徵...")
def build_input_text(row):
    return (
        "Question: " + str(row['QuestionText']).strip() + "\n"
        "Student selected: " + str(row['MC_Answer']).strip() + "\n"
        "Student explanation: " + str(row['StudentExplanation']).strip()
    )

train_df['text'] = train_df.apply(build_input_text, axis=1)
train_df['Misconception'] = train_df.get('Misconception', pd.Series([np.nan]*len(train_df))).fillna('NA')

print("Encoding texts...")
embeddings = model.encode(train_df['text'].tolist(), batch_size=128, show_progress_bar=True)
train_df['embedding'] = list(embeddings)

print("Calculating local anchors...")
anchor_df = train_df.groupby(['QuestionId', 'Misconception'])['embedding'].apply(
    lambda x: np.mean(np.vstack(x), axis=0)
).reset_index()

local_anchors = {}
for _, row in anchor_df.iterrows():
    qid = row['QuestionId']
    misc = row['Misconception']
    if qid not in local_anchors:
        local_anchors[qid] = {}
    local_anchors[qid][misc] = row['embedding']

os.makedirs("map_artifacts", exist_ok=True)
with open("map_artifacts/local_anchors.pkl", "wb") as f:
    pickle.dump(local_anchors, f)

print("局部 Anchor 建立完成，已儲存！")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

正在合併文字特徵...
Encoding texts...


Batches:   0%|          | 0/287 [00:00<?, ?it/s]

Calculating local anchors...
局部 Anchor 建立完成，已儲存！


In [3]:
import os
import random
import pickle
import gc
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader


gc.collect()
torch.cuda.empty_cache()

print("準備微調訓練資料 ...")

# 1. 前置處理：加入指令引導
def build_instruction_text(row):
    instruction = "Represent the mathematical misconception in this student's explanation for classification: "
    return instruction + str(row['text'])

train_df['instructed_text'] = train_df.apply(build_instruction_text, axis=1)

train_examples = []

# 2. 建立 QuestionId 對照表
q_misc_texts = {}
for (qid, misc), group in train_df[train_df['Misconception'] != 'NA'].groupby(['QuestionId', 'Misconception']):
    if qid not in q_misc_texts:
        q_misc_texts[qid] = {}
    q_misc_texts[qid][misc] = group['instructed_text'].tolist()

# 3. 構建 Triplet 訓練樣本
for qid, misc_dict in q_misc_texts.items():
    misc_list = list(misc_dict.keys())

    for i, misc_a in enumerate(misc_list):
        texts_pos = misc_dict[misc_a]
        if len(texts_pos) < 2:
            continue

        other_miscs = [m for m in misc_list if m != misc_a]

        for j in range(len(texts_pos) - 1):
            anchor = texts_pos[j]
            positive = texts_pos[j+1]

            if other_miscs:
                random.seed(42 + j)
                neg_misc = random.choice(other_miscs)
                hard_negative = random.choice(misc_dict[neg_misc])
                train_examples.append(InputExample(texts=[anchor, positive, hard_negative]))
            else:
                train_examples.append(InputExample(texts=[anchor, positive]))

print(f"共收集到 {len(train_examples)} 筆樣本。")


train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2) # Reduced batch_size from 4 to 2

# 定義 Loss Function
train_loss = losses.MultipleNegativesRankingLoss(model)

print("開始微調模型 (Fine-tuning)...")
output_path = 'map_artifacts/bge_finetuned'
os.makedirs(output_path, exist_ok=True)

# 5. 執行模型擬合
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    warmup_steps=100,
    output_path=output_path,
    show_progress_bar=True
)

print(f"已儲存至 {output_path}")

準備微調訓練資料 ...


Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


共收集到 9821 筆樣本。
開始微調模型 (Fine-tuning)...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.188433
1000,0.107761
1500,0.128900
2000,0.086929
2500,0.122131
3000,0.072052
3500,0.084416
4000,0.091531
4500,0.084019
5000,0.075047


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

已儲存至 map_artifacts/bge_finetuned
